In [ ]:
import sys
sys.path.append("..")
from src.temizle import temizle

print(temizle("Kanka 12.30 da ESPARKTAYIZ :)"))
print(temizle("İSTANBUL'da %50 İNDİRİM! [TEL] arayın"))

In [ ]:
import pandas as pd
import csv

In [ ]:
train = pd.read_csv(r"C:\Users\SCT\Downloads\tr_train.csv")
test = pd.read_csv(r"C:\Users\SCT\Downloads\tr_test.csv")

#VERİ ÇEKME


In [ ]:
train.shape, test.shape
#SHAPELERİ KONTROL

In [ ]:
train["ikili"].value_counts()
#DENGE KOTROLÜ

In [ ]:
train["belirsiz"].value_counts(dropna=False), test["belirsiz"].value_counts(dropna=False)


In [ ]:
#BELİRSİZLERDEN KURTULUYORUM
train = train[train["belirsiz"] != "e"]
X_train = train["mesaj"]
y_train = train["ikili"]
X_test = test["mesaj"]
y_test = test["ikili"]
print(X_train.shape, X_test.shape)
print(y_train.value_counts())

In [ ]:
#TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
print(X_train_vec.shape, X_test_vec.shape)
print(len(vec.vocabulary_))

In [ ]:
#Naif Bayes :D

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)
print(classification_report(y_test, y_pred_nb, digits = 3))

In [ ]:
hatalar = test[y_test != y_pred_nb]
hatalar[["id", "mesaj", "ikili", "kaynak_etiket"]]

In [ ]:
#HATALARI ANALİZ ETMECE

kelimeler = vec.get_feature_names_out()
fark = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

def neden(i):
    idx = X_test_vec[i].nonzero()[1]
    return sorted(zip(kelimeler[idx], fark[idx]), key=lambda t: -t[1])

print(nb.classes_)
print(neden(7))
print(neden(27))

In [ ]:
#SVM, naif fena değil ama veri seti küçük olduğundan dolayı SVM ile denemece.
from sklearn.svm import LinearSVC
svm = LinearSVC(random_state=55)
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)
print(classification_report(y_test, y_pred_svm, digits = 3))
print(test[y_test != y_pred_svm][["id", "mesaj", "ikili",]])

In [ ]:
vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))
X_train_vec_c = vec_c.fit_transform(X_train)
X_test_vec_c = vec_c.transform(X_test)
print(X_train_vec_c.shape)

for ad, model in ["NB", MultinomialNB()], ["SVM", LinearSVC(random_state=55)]:
    model.fit(X_train_vec_c, y_train)
    tahmin = model.predict(X_test_vec_c)
    print(ad)
    print(classification_report(y_test, tahmin, digits = 3))
    print(test[y_test != tahmin]["id"].tolist())

In [ ]:
train["fold"].value_counts().sort_index()

In [ ]:
#CROSS VALIDATION İÇİN PIPELINE 

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import PredefinedSplit, cross_val_score
cv = PredefinedSplit(train["fold"])

adaylar = {"NB kelime": make_pipeline(TfidfVectorizer(), MultinomialNB()),
           "SVM kelime":make_pipeline(TfidfVectorizer(), LinearSVC(random_state=55)),
           "NB harf":   make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), MultinomialNB()),
           "SVM harf" : make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), LinearSVC(random_state=55))}
for ad, model in adaylar.items():
    skor = cross_val_score(model,X_train, y_train, cv=cv, scoring="f1_macro")
    print(f"{ad}: {skor.mean():.3f} ± {skor.std():.3f} {skor.round(3)}")



In [ ]:
import joblib
from pathlib import Path

final = adaylar["SVM harf"]
final.fit(X_train, y_train)
print(classification_report(y_test, final.predict(X_test), digits = 3))

Path("../models").mkdir(exist_ok=True)
joblib.dump(final, "../models/svm_char_tr.joblib")

yuklenen = joblib.load("../models/svm_char_tr.joblib")
print(yuklenen.predict(["kanka aksam maca geliyor musun"
                        , "Tebrikler! 5000 TL kazandiniz, "
                        "hemen tiklayin"]))


In [ ]:
#temizle.py append SONRASI SON TEST
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV

spam_f1 = make_scorer(f1_score, pos_label="spam")

def kelime():
    return TfidfVectorizer(preprocessor=temizle)

def harf():
    return TfidfVectorizer(preprocessor=temizle, analyzer="char_wb", ngram_range=(3, 5))

nb_izgara = {"multinomialnb__alpha": [0.01, 0.1, 0.5, 1.0]}
svm_izgara = {"linearsvc__C": [0.1, 0.5, 1, 5, 10]}

adaylar = {
    "NB kelime":  (make_pipeline(kelime(), MultinomialNB()), nb_izgara),
    "SVM kelime": (make_pipeline(kelime(), LinearSVC(random_state=42)), svm_izgara),
    "NB harf":    (make_pipeline(harf(), MultinomialNB()), nb_izgara),
    "SVM harf":   (make_pipeline(harf(), LinearSVC(random_state=42)), svm_izgara),
}

sonuclar = {}
for ad, (model, izgara) in adaylar.items():
    gs = GridSearchCV(model, izgara, cv=cv, scoring=spam_f1)
    gs.fit(X_train, y_train)
    std = gs.cv_results_["std_test_score"][gs.best_index_]
    print(f"{ad}: {gs.best_score_:.3f} ± {std:.3f}   {gs.best_params_}")
    sonuclar[ad] = gs

In [ ]:
genis = {
    "NB kelime":  {"multinomialnb__alpha": [0.001, 0.003, 0.01, 0.03]},
    "SVM kelime": {"linearsvc__C": [5, 10, 30, 100]},
}

for ad, izgara in genis.items():
    model = adaylar[ad][0]
    gs = GridSearchCV(model, izgara, cv=cv, scoring=spam_f1)
    gs.fit(X_train, y_train)
    std = gs.cv_results_["std_test_score"][gs.best_index_]
    print(f"{ad}: {gs.best_score_:.3f} ± {std:.3f}   {gs.best_params_}")
    sonuclar[ad] = gs

In [ ]:
from sklearn.metrics import precision_score, recall_score

for ad, gs in sonuclar.items():
    tahmin = gs.predict(X_test)
    f1 = f1_score(y_test, tahmin, pos_label="spam")
    p = precision_score(y_test, tahmin, pos_label="spam")
    r = recall_score(y_test, tahmin, pos_label="spam")
    hatalar = test[y_test != tahmin]["id"].tolist()
    print(f"{ad}: F1 {f1:.3f}  P {p:.3f}  R {r:.3f}  hatalar {hatalar}")

In [ ]:
#SON KEZ MODELİ TEKRAR KAYDEDİYOREE 
final = sonuclar["SVM harf"].best_estimator_
joblib.dump(final, "../models/svm_char_tr.joblib")

yuklenen = joblib.load("../models/svm_char_tr.joblib")
print(yuklenen.predict(["Kanka 12.30 da esparktayiz", "Degerli musterimiz kartiniz bloke oldu, [URL] tiklayin"]))

In [ ]:
vec = final.named_steps["tfidfvectorizer"]
svm = final.named_steps["linearsvc"]
mesaj = "Degerli musterimiz kartiniz bloke oldu, [URL] tiklayin"

print("skor:", svm.decision_function(vec.transform([mesaj])))

x = vec.transform([mesaj])
idx = x.nonzero()[1]
katki = x.toarray()[0][idx] * svm.coef_[0][idx]
parcalar = vec.get_feature_names_out()[idx]
sirali = sorted(zip(parcalar, katki.round(3)), key=lambda t: t[1])

print("meşruya iten:", sirali[:8])
print("spam'e iten:", sirali[-8:])
print("train'de [URL] geçen mesaj:", train["mesaj"].str.contains(r"\[URL\]").sum())

In [ ]:
maskeler = train["mesaj"].str.findall(r"\[[A-Z_]+\]").explode().value_counts()
print(maskeler)
print(train["mesaj"].str.contains("http|www", case=False).sum())

In [ ]:
from src.maskele import maskele

denemeler = [
    "Degerli musterimiz kartiniz bloke oldu, [LINK] tiklayin",
    "Degerli musterimiz kartiniz bloke oldu, http://garanti-guvenlik.xyz tiklayin",
]

for m in denemeler:
    m2 = maskele(m)
    skor = final.decision_function([m2])[0]
    print(f"{skor:+.3f}  {final.predict([m2])[0]}  |  {m2}")